# Without PCA LDA

In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import scipy.stats as st

# Load the dataset
data = pd.read_csv("/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/review/finalDatasetWithUPDRSScore.csv")

# Define non-feature, numerical, and categorical columns
non_feature_columns = ['Patient ID', 'Visit Date', 'UPDRS_SCORE', 'Visit', 'Visit_int',
                       'NHY', 'DATSCAN_PUTAMEN_R', 'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
                       'DATSCAN_PUTAMEN_L_ANT', 'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L']
numerical_features = [
    'Area', 'Circularity', 'ConvexArea', 'EquivDiameter', 'Extent',
    'FilledArea', 'Kurtosis', 'Major axis length', 'Mean',
    'Minor axis length', 'PA_ratio', 'Shannon_Entropy', 'Skewness',
    'Solidity', 'Standard Deviation', 'brightness', 'contrast',
    'correlation', 'dissimilarity', 'energy', 'gabor_energy',
    'gabor_entropy', 'homogeneity', 'lbp_energy', 'lbp_entropy'
]
categorical_features = [
    'NP1ANXS', 'NP1APAT', 'NP1COG', 'NP1DDS', 'NP1DPRS',
    'NP1HALL', 'NP1CNST', 'NP1FATG', 'NP1LTHD', 'NP1PAIN', 'NP1SLPD',
    'NP1SLPN', 'NP1URIN', 'NP2DRES', 'NP2EAT', 'NP2FREZ', 'NP2HOBB',
    'NP2HWRT', 'NP2HYGN', 'NP2RISE', 'NP2SALV', 'NP2SPCH', 'NP2SWAL',
    'NP2TRMR', 'NP2TURN', 'NP2WALK', 'NP3BRADY', 'NP3FACXP', 'NP3FRZGT',
    'NP3FTAPL', 'NP3FTAPR', 'NP3GAIT', 'NP3HMOVL', 'NP3HMOVR', 'NP3KTRML',
    'NP3KTRMR', 'NP3LGAGL', 'NP3LGAGR', 'NP3POSTR', 'NP3PRSPL', 'NP3PRSPR',
    'NP3PSTBL', 'NP3PTRML', 'NP3PTRMR', 'NP3RIGLL', 'NP3RIGLU', 'NP3RIGN',
    'NP3RIGRL', 'NP3RIGRU', 'NP3RISNG', 'NP3RTALJ', 'NP3RTALL', 'NP3RTALU',
    'NP3RTARL', 'NP3RTARU', 'NP3RTCON', 'NP3SPCH', 'NP3TTAPL', 'NP3TTAPR'
]
target_column = "Disease_Severity"

# Preprocessing pipeline for numerical features
scaler = MinMaxScaler()
data[numerical_features] = scaler.fit_transform(data[numerical_features])

# Split data into X (features) and y (target)
X = data[numerical_features + categorical_features]
y = data[target_column]

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Binarize the output for multi-class AUC calculation
y_test_binarized = label_binarize(y_test, classes=np.unique(y))
n_classes = y_test_binarized.shape[1]

# Function to calculate 95% confidence interval using binomial proportion
def calculate_ci_binomial(accuracy, n_samples, confidence=0.95):
    """
    Calculate Wilson score confidence interval for classification accuracy.
    This is more accurate than normal approximation for proportions.
    """
    z = st.norm.ppf((1 + confidence) / 2)  # 1.96 for 95% CI
    p = accuracy
    n = n_samples
    
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    
    ci_lower = center - margin
    ci_upper = center + margin
    
    return ci_lower, ci_upper

# Define classifiers including ensemble methods
classifiers = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42),
    'Naïve Bayes': GaussianNB(),
    'XGB': XGBClassifier(random_state=42),
    # Advanced Ensemble Methods
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Bagging (DT)': BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42),
    'Voting (Hard)': VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42))
        ],
        voting='hard'
    ),
    'Voting (Soft)': VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42))
        ],
        voting='soft'
    ),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42)),
            ('et', ExtraTreesClassifier(random_state=42))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5
    )
}

# Initialize results storage
results_list = []

# Train and evaluate each classifier using original features
for name, clf in classifiers.items():
    print(f"\nTraining {name}...")
    clf.fit(X_train, y_train)
    y_train_pred = clf.predict(X_train)
    y_test_pred = clf.predict(X_test)
    
    # Calculate metrics for training set
    train_accuracy = accuracy_score(y_train, y_train_pred)
    
    # Calculate metrics for testing set
    test_accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, average='weighted')
    recall = recall_score(y_test, y_test_pred, average='weighted')
    f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Calculate AUC (ROC AUC for multi-class)
    try:
        if hasattr(clf, "predict_proba"):
            y_test_proba = clf.predict_proba(X_test)
            auc_score = roc_auc_score(y_test_binarized, y_test_proba, average='weighted', multi_class='ovr')
        elif hasattr(clf, "decision_function"):
            y_test_decision = clf.decision_function(X_test)
            auc_score = roc_auc_score(y_test_binarized, y_test_decision, average='weighted', multi_class='ovr')
        else:
            auc_score = np.nan
    except Exception as e:
        auc_score = np.nan
    
    # Calculate Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_test, y_test_pred)
    
    # Calculate Cohen's Kappa
    kappa = cohen_kappa_score(y_test, y_test_pred)
    
    # Calculate 95% Confidence Interval for Accuracy using Wilson Score method
    ci_lower, ci_upper = calculate_ci_binomial(test_accuracy, len(y_test))
    
    # Store results in dictionary
    results_dict = {
        'Classifier': name,
        'Training_Accuracy': train_accuracy,
        'Testing_Accuracy': test_accuracy,
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'AUC_ROC': auc_score,
        'MCC': mcc,
        'Cohen_Kappa': kappa
    }
    
    results_list.append(results_dict)
    
    # Print metrics
    print(f"\n{'='*60}")
    print(f"{name} Results:")
    print(f"{'='*60}")
    print(f"Training Accuracy:     {train_accuracy:.4f}")
    print(f"Testing Accuracy:      {test_accuracy:.4f}")
    print(f"Accuracy 95% CI:       [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")
    print(f"F1-score:              {f1:.4f}")
    print(f"AUC (ROC):             {auc_score if isinstance(auc_score, str) or np.isnan(auc_score) else f'{auc_score:.4f}'}")
    print(f"Matthews Corr. Coef:   {mcc:.4f}")
    print(f"Cohen's Kappa:         {kappa:.4f}")
    print(f"{'='*60}")

# Convert results to DataFrame
results_df = pd.DataFrame(results_list)

# Save to CSV
output_file = "/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/review/classifier_results_fixed.csv"
results_df.to_csv(output_file, index=False)
print(f"\n\nResults saved to: {output_file}")

# Display summary
print("\n\nSummary of Results:")
print(results_df.to_string(index=False))



Training Random Forest...

Random Forest Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6460
Accuracy 95% CI:       [0.5877, 0.7002]
Precision:             0.6911
Recall:                0.6460
F1-score:              0.6115
AUC (ROC):             0.8756
Matthews Corr. Coef:   0.4901
Cohen's Kappa:         0.4804

Training Logistic Regression...

Logistic Regression Results:
Training Accuracy:     0.9953
Testing Accuracy:      0.8358
Accuracy 95% CI:       [0.7873, 0.8749]
Precision:             0.8436
Recall:                0.8358
F1-score:              0.8298
AUC (ROC):             0.9792
Matthews Corr. Coef:   0.7742
Cohen's Kappa:         0.7709

Training AdaBoost...

Random Forest Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6460
Accuracy 95% CI:       [0.5877, 0.7002]
Precision:             0.6911
Recall:                0.6460
F1-score:              0.6115
AUC (ROC):             0.8756
Matthews Corr. Coef:   0.4901
Cohen's Kappa:         0.4804


/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



AdaBoost Results:
Training Accuracy:     0.5878
Testing Accuracy:      0.5620
Accuracy 95% CI:       [0.5028, 0.6195]
Precision:             0.5031
Recall:                0.5620
F1-score:              0.5166
AUC (ROC):             0.7362
Matthews Corr. Coef:   0.3696
Cohen's Kappa:         0.3594

Training Decision Tree...

Decision Tree Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.4453
Accuracy 95% CI:       [0.3876, 0.5045]
Precision:             0.4567
Recall:                0.4453
F1-score:              0.4452
AUC (ROC):             0.6198
Matthews Corr. Coef:   0.2438
Cohen's Kappa:         0.2427

Training Gradient Boosting...

Gradient Boosting Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6569
Accuracy 95% CI:       [0.5989, 0.7106]
Precision:             0.6552
Recall:                0.6569
F1-score:              0.6365
AUC (ROC):             0.8831
Matthews Corr. Coef:   0.5140
Cohen's Kappa:         0.5101

Training KNN...

KNN Results:


/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parame


Bagging (DT) Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6606
Accuracy 95% CI:       [0.6026, 0.7141]
Precision:             0.6502
Recall:                0.6606
F1-score:              0.6247
AUC (ROC):             0.8816
Matthews Corr. Coef:   0.5127
Cohen's Kappa:         0.5052

Training Voting (Hard)...

Voting (Hard) Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6752
Accuracy 95% CI:       [0.6176, 0.7279]
Precision:             0.6961
Recall:                0.6752
F1-score:              0.6492
AUC (ROC):             nan
Matthews Corr. Coef:   0.5363
Cohen's Kappa:         0.5297

Training Voting (Soft)...

Voting (Hard) Results:
Training Accuracy:     1.0000
Testing Accuracy:      0.6752
Accuracy 95% CI:       [0.6176, 0.7279]
Precision:             0.6961
Recall:                0.6752
F1-score:              0.6492
AUC (ROC):             nan
Matthews Corr. Coef:   0.5363
Cohen's Kappa:         0.5297

Training Voting (Soft)...

Voting (Soft)

In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, matthews_corrcoef, cohen_kappa_score
from sklearn.preprocessing import label_binarize
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, ExtraTreesClassifier, BaggingClassifier
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import scipy.stats as st

# Load the dataset
data = pd.read_csv("/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/review/finalDatasetWithUPDRSScore.csv")

# Define non-feature, numerical, and categorical columns
non_feature_columns = ['Patient ID', 'Visit Date', 'UPDRS_SCORE', 'Visit', 'Visit_int',
                       'NHY', 'DATSCAN_PUTAMEN_R', 'DATSCAN_CAUDATE_R', 'DATSCAN_CAUDATE_L',
                       'DATSCAN_PUTAMEN_L_ANT', 'DATSCAN_PUTAMEN_R_ANT', 'DATSCAN_PUTAMEN_L']
numerical_features = [
    'Area', 'Circularity', 'ConvexArea', 'EquivDiameter', 'Extent',
    'FilledArea', 'Kurtosis', 'Major axis length', 'Mean',
    'Minor axis length', 'PA_ratio', 'Shannon_Entropy', 'Skewness',
    'Solidity', 'Standard Deviation', 'brightness', 'contrast',
    'correlation', 'dissimilarity', 'energy', 'gabor_energy',
    'gabor_entropy', 'homogeneity', 'lbp_energy', 'lbp_entropy'
]
categorical_features = [
    'NP1ANXS', 'NP1APAT', 'NP1COG', 'NP1DDS', 'NP1DPRS',
    'NP1HALL', 'NP1CNST', 'NP1FATG', 'NP1LTHD', 'NP1PAIN', 'NP1SLPD',
    'NP1SLPN', 'NP1URIN', 'NP2DRES', 'NP2EAT', 'NP2FREZ', 'NP2HOBB',
    'NP2HWRT', 'NP2HYGN', 'NP2RISE', 'NP2SALV', 'NP2SPCH', 'NP2SWAL',
    'NP2TRMR', 'NP2TURN', 'NP2WALK', 'NP3BRADY', 'NP3FACXP', 'NP3FRZGT',
    'NP3FTAPL', 'NP3FTAPR', 'NP3GAIT', 'NP3HMOVL', 'NP3HMOVR', 'NP3KTRML',
    'NP3KTRMR', 'NP3LGAGL', 'NP3LGAGR', 'NP3POSTR', 'NP3PRSPL', 'NP3PRSPR',
    'NP3PSTBL', 'NP3PTRML', 'NP3PTRMR', 'NP3RIGLL', 'NP3RIGLU', 'NP3RIGN',
    'NP3RIGRL', 'NP3RIGRU', 'NP3RISNG', 'NP3RTALJ', 'NP3RTALL', 'NP3RTALU',
    'NP3RTARL', 'NP3RTARU', 'NP3RTCON', 'NP3SPCH', 'NP3TTAPL', 'NP3TTAPR'
]
target_column = "Disease_Severity"

# Preprocessing pipeline for numerical features
scaler = MinMaxScaler()
data[numerical_features] = scaler.fit_transform(data[numerical_features])

# Split data into X (features) and y (target)
X = data[numerical_features + categorical_features]
y = data[target_column]

# Encode the target variable
le = LabelEncoder()
y = le.fit_transform(y)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Apply LDA for dimensionality reduction
n_components = 2  # You can use: min(X.shape[1], len(np.unique(y)) - 1)
print(f"Number of LDA components: {n_components}")

lda = LDA(n_components=n_components)
X_train_lda = lda.fit_transform(X_train, y_train)
X_test_lda = lda.transform(X_test)

print(f"Original training data shape: {X_train.shape}")
print(f"LDA transformed training data shape: {X_train_lda.shape}")
print(f"Explained variance ratio: {lda.explained_variance_ratio_}")

# Binarize the output for multi-class AUC calculation
y_test_binarized = label_binarize(y_test, classes=np.unique(y))
n_classes = y_test_binarized.shape[1]

# Function to calculate 95% confidence interval using Wilson score method
def calculate_ci_binomial(accuracy, n_samples, confidence=0.95):
    """
    Calculate Wilson score confidence interval for classification accuracy.
    This is more accurate than normal approximation for proportions.
    """
    z = st.norm.ppf((1 + confidence) / 2)  # 1.96 for 95% CI
    p = accuracy
    n = n_samples
    
    denominator = 1 + z**2 / n
    center = (p + z**2 / (2 * n)) / denominator
    margin = z * np.sqrt((p * (1 - p) / n + z**2 / (4 * n**2))) / denominator
    
    ci_lower = center - margin
    ci_upper = center + margin
    
    return ci_lower, ci_upper

# Define classifiers including ensemble methods
classifiers = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'AdaBoost': AdaBoostClassifier(random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(probability=True, random_state=42),
    'Naïve Bayes': GaussianNB(),
    'XGB': XGBClassifier(random_state=42),
    'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
    'Bagging (DT)': BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, random_state=42),
    'Voting (Hard)': VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42))
        ],
        voting='hard'
    ),
    'Voting (Soft)': VotingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42))
        ],
        voting='soft'
    ),
    'Stacking': StackingClassifier(
        estimators=[
            ('rf', RandomForestClassifier(random_state=42)),
            ('gb', GradientBoostingClassifier(random_state=42)),
            ('xgb', XGBClassifier(random_state=42)),
            ('et', ExtraTreesClassifier(random_state=42))
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5
    )
}

# Initialize results storage
results_list = []

# Train and evaluate each classifier using LDA-transformed features
for name, clf in classifiers.items():
    print(f"\nTraining {name} with LDA features...")
    clf.fit(X_train_lda, y_train)
    y_train_pred = clf.predict(X_train_lda)
    y_test_pred = clf.predict(X_test_lda)
    
    # Calculate metrics for training set
    train_accuracy = accuracy_score(y_train, y_train_pred)
    
    # Calculate metrics for testing set
    test_accuracy = accuracy_score(y_test, y_test_pred)
    precision = precision_score(y_test, y_test_pred, average='weighted')
    recall = recall_score(y_test, y_test_pred, average='weighted')
    f1 = f1_score(y_test, y_test_pred, average='weighted')
    
    # Calculate AUC (ROC AUC for multi-class)
    try:
        if hasattr(clf, "predict_proba"):
            y_test_proba = clf.predict_proba(X_test_lda)
            auc_score = roc_auc_score(y_test_binarized, y_test_proba, average='weighted', multi_class='ovr')
        elif hasattr(clf, "decision_function"):
            y_test_decision = clf.decision_function(X_test_lda)
            auc_score = roc_auc_score(y_test_binarized, y_test_decision, average='weighted', multi_class='ovr')
        else:
            auc_score = np.nan
    except Exception as e:
        auc_score = np.nan
    
    # Calculate Matthews Correlation Coefficient (MCC)
    mcc = matthews_corrcoef(y_test, y_test_pred)
    
    # Calculate Cohen's Kappa
    kappa = cohen_kappa_score(y_test, y_test_pred)
    
    # Calculate 95% Confidence Interval for Accuracy using Wilson Score method
    ci_lower, ci_upper = calculate_ci_binomial(test_accuracy, len(y_test))
    
    # Store results in dictionary
    results_dict = {
        'Classifier': name,
        'Training_Accuracy': train_accuracy,
        'Testing_Accuracy': test_accuracy,
        'CI_Lower': ci_lower,
        'CI_Upper': ci_upper,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'AUC_ROC': auc_score,
        'MCC': mcc,
        'Cohen_Kappa': kappa
    }
    
    results_list.append(results_dict)
    
    # Print metrics
    print(f"\n{'='*60}")
    print(f"{name} Results (with LDA):")
    print(f"{'='*60}")
    print(f"Training Accuracy:     {train_accuracy:.4f}")
    print(f"Testing Accuracy:      {test_accuracy:.4f}")
    print(f"Accuracy 95% CI:       [{ci_lower:.4f}, {ci_upper:.4f}]")
    print(f"Precision:             {precision:.4f}")
    print(f"Recall:                {recall:.4f}")
    print(f"F1-score:              {f1:.4f}")
    print(f"AUC (ROC):             {auc_score if isinstance(auc_score, str) or np.isnan(auc_score) else f'{auc_score:.4f}'}")
    print(f"Matthews Corr. Coef:   {mcc:.4f}")
    print(f"Cohen's Kappa:         {kappa:.4f}")
    print(f"{'='*60}")

# Convert results to DataFrame
results_df = pd.DataFrame(results_list)

# Save to CSV
output_file = "/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/review/classifier_results_lda_fixed.csv"
results_df.to_csv(output_file, index=False)
print(f"\n\nResults saved to: {output_file}")

# Display summary
print("\n\nSummary of Results (with LDA):")
print(results_df.to_string(index=False))

# Print top contributing features for each LDA component
print("\n\nTop 10 features for each LDA component:")
for i in range(n_components):
    component_features = np.abs(lda.scalings_[:, i])
    top_10_idx = component_features.argsort()[::-1][:10]
    top_10_features = X.columns[top_10_idx]
    print(f"\nLD{i+1}:")
    for j, feature in enumerate(top_10_features, 1):
        print(f"  {j}. {feature} (coefficient: {lda.scalings_[top_10_idx[j-1], i]:.4f})")


Number of LDA components: 2
Original training data shape: (638, 84)
LDA transformed training data shape: (638, 2)
Explained variance ratio: [0.92616022 0.03339927]

Training Random Forest with LDA features...

Random Forest Results (with LDA):
Training Accuracy:     1.0000
Testing Accuracy:      0.8978
Accuracy 95% CI:       [0.8563, 0.9283]
Precision:             0.8991
Recall:                0.8978
F1-score:              0.8966
AUC (ROC):             0.9625
Matthews Corr. Coef:   0.8594
Cohen's Kappa:         0.8587

Training Logistic Regression with LDA features...

Logistic Regression Results (with LDA):
Training Accuracy:     0.9514
Testing Accuracy:      0.8905
Accuracy 95% CI:       [0.8480, 0.9222]
Precision:             0.8945
Recall:                0.8905
F1-score:              0.8878
AUC (ROC):             0.9863
Matthews Corr. Coef:   0.8497
Cohen's Kappa:         0.8481

Training AdaBoost with LDA features...

AdaBoost Results (with LDA):
Training Accuracy:     0.8401
Test

/home/m8m/Projects/PPMI_Research_on_Parkinsons-master/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



Gradient Boosting Results (with LDA):
Training Accuracy:     1.0000
Testing Accuracy:      0.8723
Accuracy 95% CI:       [0.8275, 0.9067]
Precision:             0.8769
Recall:                0.8723
F1-score:              0.8716
AUC (ROC):             0.9758
Matthews Corr. Coef:   0.8244
Cohen's Kappa:         0.8235

Training KNN with LDA features...

KNN Results (with LDA):
Training Accuracy:     0.9561
Testing Accuracy:      0.9051
Accuracy 95% CI:       [0.8646, 0.9344]
Precision:             0.9079
Recall:                0.9051
F1-score:              0.9043
AUC (ROC):             0.9614
Matthews Corr. Coef:   0.8696
Cohen's Kappa:         0.8686

Training SVM with LDA features...

SVM Results (with LDA):
Training Accuracy:     0.9561
Testing Accuracy:      0.9051
Accuracy 95% CI:       [0.8646, 0.9344]
Precision:             0.9097
Recall:                0.9051
F1-score:              0.9041
AUC (ROC):             0.9889
Matthews Corr. Coef:   0.8698
Cohen's Kappa:         0.8687

